In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import  pandas as pd 
import json 
import os
from glob import glob
import seaborn as sns 
import numpy as np 
import re
import tikzplotly
import plotly.express as px
from IPython.display import display
from PIL import Image
import matplotlib as mpl
import matplotlib.pyplot as plt 
import plotly 
import plotly.graph_objects as go

from IPython.display import IFrame

from utils.benchmark import * 

In [4]:
notebook_name="02-c5.large-geobft-no-cluster-commit"
os.makedirs(f"outputs/{notebook_name}", exist_ok=True)


import zipfile
import datetime

timestamp=datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

with zipfile.ZipFile(f'outputs/{notebook_name}/notebook_files_{timestamp}.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(f'outputs/{notebook_name}'):
        for file in files:
            if not file.endswith('.zip'):
                zipf.write(os.path.join(root, file), 
                           os.path.relpath(os.path.join(root, file), 
                                           os.path.join(f'outputs/{notebook_name}', '..')))
                os.remove(os.path.join(root, file))


In [5]:
# folder="../../aws/benchmark/out/modubft/full_spread_modubft_with_bytes_sent/" 
folder = "../../aws/benchmark/out/geobft/final-geobft-not-alternating-non-optimistic/"
os.listdir(folder)

['20251029160150-cb64-v512',
 '20251029160301-cb64-v4096',
 '20251029160409-cb128-v512',
 '20251029160517-cb128-v4096',
 '20251029160623-cb256-v512',
 '20251029160728-cb256-v4096',
 '20251029161648-cb64-v512',
 '20251029161740-cb64-v4096',
 '20251029161828-cb128-v512',
 '20251029161915-cb128-v4096',
 '20251029161959-cb256-v512',
 '20251029162046-cb256-v4096',
 '20251029162833-cb64-v512',
 '20251029162923-cb64-v4096',
 '20251029163011-cb128-v512',
 '20251029163058-cb128-v4096',
 '20251029163146-cb256-v512',
 '20251029163233-cb256-v4096',
 '20251029163948-cb64-v512',
 '20251029164028-cb64-v4096',
 '20251029164108-cb128-v512',
 '20251029164145-cb128-v4096',
 '20251029164225-cb256-v512',
 '20251029164303-cb256-v4096',
 '20251030135938-cb64-v512',
 '20251030140044-cb64-v4096',
 '20251030140148-cb128-v512',
 '20251030140249-cb128-v4096',
 '20251030165441-cb1-v512',
 '20251030165530-cb1-v4096',
 '20251030165612-cb16-v512',
 '20251030165656-cb16-v4096',
 '20251030222703-cb1-v512',
 '2025103022

In [6]:
benchmarks = glob(f"{folder}*/")
benchmarks += glob("../../aws/benchmark/out/geobft/20251025-224449-large/*/")  


In [7]:
benchmarks

['../../aws/benchmark/out/geobft/final-geobft-not-alternating-non-optimistic/20251029160150-cb64-v512/',
 '../../aws/benchmark/out/geobft/final-geobft-not-alternating-non-optimistic/20251029160301-cb64-v4096/',
 '../../aws/benchmark/out/geobft/final-geobft-not-alternating-non-optimistic/20251029160409-cb128-v512/',
 '../../aws/benchmark/out/geobft/final-geobft-not-alternating-non-optimistic/20251029160517-cb128-v4096/',
 '../../aws/benchmark/out/geobft/final-geobft-not-alternating-non-optimistic/20251029160623-cb256-v512/',
 '../../aws/benchmark/out/geobft/final-geobft-not-alternating-non-optimistic/20251029160728-cb256-v4096/',
 '../../aws/benchmark/out/geobft/final-geobft-not-alternating-non-optimistic/20251029161648-cb64-v512/',
 '../../aws/benchmark/out/geobft/final-geobft-not-alternating-non-optimistic/20251029161740-cb64-v4096/',
 '../../aws/benchmark/out/geobft/final-geobft-not-alternating-non-optimistic/20251029161828-cb128-v512/',
 '../../aws/benchmark/out/geobft/final-geobft-

In [8]:
throughputs = process_throughput_benchmarks(benchmarks)
throughputs["num_peers"] = throughputs["num_peers"]/throughputs["number_of_clusters"]
throughputs.drop_duplicates(["num_peers","vallen","cluster_batch_size","nonoptimistic","alternating","number_of_clusters"], inplace=True,keep="last")

In [9]:
grouped = throughputs.query("cluster_batch_size < 256").groupby(["num_peers","vallen"])

In [10]:
for gr, g in grouped:
    fig  = go.Figure()
    for v,data in g.sort_values(["nonoptimistic","cluster_batch_size","vallen","number_of_clusters"]).groupby(["number_of_clusters","nonoptimistic"]):
        
        fig.add_trace(go.Scatter(
            name=f"Num Clusters={v[0]}- {'Non-Optimistic' if v[1] else 'Optimistic'}",
            x=data['cluster_batch_size'],
            y=data['throughput']
        ))
    #         xlabel="Cluster Batch Size",
    # ylabel="Throughput (ops/sec)",
    fig.update_layout(
        title=f"GeoBFT Throughput - Num Peers per Cluster: {gr[0]}, payload size: {gr[1]} bytes",
        xaxis_title="Cluster Batch Size",
        yaxis_title="Throughput (ops/sec)",
        barmode='group'
    )
    fig.show()
        
    tikzplotly.save(filepath=f"outputs/{notebook_name}/geobft-throughput-numpeers-{gr}.tex",fig=fig)
    

/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



In [11]:
processed_cpu_usage = process_cpu_usage(benchmarks)
processed_cpu_usage["num_peers"] = processed_cpu_usage["num_peers"]/processed_cpu_usage["number_of_clusters"]
# processed_cpu_usage.drop_duplicates(["num_peers","vallen","cluster_batch_size","nonoptimistic","alternating","number_of_clusters"], inplace=True,keep="last")


In [12]:
processed_cpu_usage

,benchmark,num_peers,vallen,runtime,cpu_usage,role,cluster_batch_size,number_of_clusters,alternating,nonoptimistic
0,../../aws/benchmark/out/geobft/final-geobft-no...,6.0,512,None,93.0,2,64,4,False,True
1,../../aws/benchmark/out/geobft/final-geobft-no...,6.0,512,None,94.0,2,64,4,False,True
2,../../aws/benchmark/out/geobft/final-geobft-no...,6.0,512,None,62.0,1,64,4,False,True
3,../../aws/benchmark/out/geobft/final-geobft-no...,6.0,512,None,62.0,1,64,4,False,True
4,../../aws/benchmark/out/geobft/final-geobft-no...,6.0,512,None,65.0,1,64,4,False,True
...,...,...,...,...,...,...,...,...,...,...
2407,../../aws/benchmark/out/geobft/20251025-224449...,3.0,4096,None,51.0,1,128,2,False,False
2408,../../aws/benchmark/out/geobft/20251025-224449...,3.0,4096,None,54.0,1,128,2,False,False
2409,../../aws/benchmark/out/geobft/20251025-224449...,3.0,4096,None,54.0,1,128,2,False,False
2410,../../aws/benchmark/out/geobft/20251025-224449...,3.0,4096,None,17.0,0,128,2,False,False


In [13]:
grouped_cpu = processed_cpu_usage.query("cluster_batch_size < 256 and role > 0 ").groupby(["num_peers","vallen","role"])

In [ ]:
for gr, g in grouped_cpu:
    fig  = go.Figure()
    for v,data in g.sort_values(["nonoptimistic","cluster_batch_size","vallen","number_of_clusters"]).groupby(["number_of_clusters","nonoptimistic","vallen"]):
        # dashed if payload size is 4096 
        fig.add_trace(go.Scatter(
            name=f"Num Clusters={v[0]}- {'Non-Optimistic' if v[1] else 'Optimistic'}",
            y=data.groupby(["role","cluster_batch_size"])['cpu_usage'].mean(),
            x=data.groupby(["role","cluster_batch_size"])['cluster_batch_size'].mean(),
            mode='lines+markers',
            line=dict(dash='dash' if v[1] else 'solid')
        ))
    fig.update_layout(
        title=f"GeoBFT CPU Usage Role: {gr[2]} - Num Peers per Cluster: {gr[0]}, payload size: {gr[1]} bytes",
        xaxis_title="Cluster Batch Size",
        yaxis_title="CPU Usage in Percent",
        barmode='group'
    )
    fig.show()
    tikzplotly.save(filepath=f"outputs/{notebook_name}/geobft-cpu-role-{gr}.tex",fig=fig)
    


/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



In [15]:
bytes_sent = process_bytes_sent(benchmarks)
bytes_sent = bytes_sent.query("cluster_batch_size < 256 and role > 0 ")
bytes_sent["num_peers"] = bytes_sent["num_peers"]/bytes_sent["number_of_clusters"]
bytes_sent_in_gbps = (bytes_sent.groupby(["num_peers","vallen","role","number_of_clusters","cluster_batch_size","benchmark","nonoptimistic"])[["bytes_sent"]].sum() * 8 * 10**(-9) * 15**(-1)).reset_index()
bytes_sent_in_gbps["bytes_sent"] = bytes_sent_in_gbps["bytes_sent"] / bytes_sent_in_gbps["number_of_clusters"]
grouped = bytes_sent_in_gbps.groupby(["num_peers","vallen","role"])

In [16]:
for gr, g in grouped:
    fig  = go.Figure()
    for v,data in g.sort_values(["nonoptimistic","cluster_batch_size","vallen","number_of_clusters"]).groupby(["number_of_clusters","nonoptimistic","vallen"]):
        # dashed if payload size is 4096 
        # display(data)
        fig.add_trace(go.Scatter(
            name=f"Num Clusters={v[0]}- {'Non-Optimistic' if v[1] else 'Optimistic'}",
            y=data.groupby(["role","cluster_batch_size"])['bytes_sent'].mean(),
            x=data.groupby(["role","cluster_batch_size"])['cluster_batch_size'].mean(),
            mode='lines+markers',
            line=dict(dash='dash' if v[1] else 'solid')
        ))
    fig.update_layout(
        title=f"GeoBFT Bytes Sent Role: {gr[2]} - Num Peers per Cluster: {gr[0]}, payload size: {gr[1]} bytes",
        xaxis_title="Cluster Batch Size",
        yaxis_title="Bytes Sent in Gbps",
        barmode='group'
    )
    tikzplotly.save(filepath=f"outputs/{notebook_name}/geobft-bytes-sent-numpeers-{gr}.tex",fig=fig)
    fig.show()

/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



In [17]:
bytes_sent = add_combined_column(bytes_sent, ['num_peers', 'vallen','number_of_clusters',"cluster_batch_size","nonoptimistic"], 'num_peers_vallen')
bytes_sent["cluster_num_peers"] = bytes_sent["num_peers"] /  bytes_sent["number_of_clusters"] 

# role_df = get_role_df(bytes_sent, role=2)
# role_df = add_combined_column(role_df, ['num_peers', 'vallen','number_of_clusters',"cluster_batch_size"], 'num_peers_vallen')
# role_df["cluster_num_peers"] = role_df["num_peers"] /  role_df["number_of_clusters"] 




In [18]:
bytes_sent = process_bytes_sent(benchmarks)
bytes_sent= bytes_sent.query("cluster_batch_size == 128")
bytes_sent["cluster_num_peers"] = bytes_sent["num_peers"] /  bytes_sent["number_of_clusters"] 
for (group,df) in bytes_sent.groupby(['cluster_num_peers']): 
    # display(df)
    # continue
    role_df = get_role_df(df, role=2)
    role_df = add_combined_column(role_df, ['vallen','number_of_clusters',"nonoptimistic"], 'num_peers_vallen')
    role_df_complete = complete_multiindex(role_df, ['typ', 'vallen','number_of_clusters',"num_peers_vallen"])

    role_df_complete = add_percent_column(role_df_complete, 'num_peers_vallen', 'bytes_sent', 'bytes_sent_percent')
    cluster_num_peers = group[0]
    print(f"Cluster Num Peers: {cluster_num_peers}")
    
     


    plot_tikz(
        role_df_complete,
        x_col="num_peers_vallen",
        y_col="bytes_sent_percent",
        cat_col="typ",
        filename=f"outputs/{notebook_name}/geobft-spread-over-azs-bytes-sent-role2-{cluster_num_peers}.tex",
        xlabel="(Number of Peers, Vallen)",
        ylabel="Bytes Sent Percent",
        bar=True,
        stack=True,
        symbolic_x=True,
        sort_x_key=lambda x: (int(x.split(",")[0][1:]),int(x.split(",")[1]),str(x.split(",")[2:-1])),
        output_dir=f"outputs/{notebook_name}/"
        
    )
     

    role_df = get_role_df(df, role=1)
    role_df = add_combined_column(role_df, ['vallen','number_of_clusters',"nonoptimistic"], 'num_peers_vallen')
    role_df_complete = complete_multiindex(role_df, ['typ', 'vallen','number_of_clusters',"num_peers_vallen"])

    role_df_complete = add_percent_column(role_df_complete, 'num_peers_vallen', 'bytes_sent', 'bytes_sent_percent')
    plot_tikz(
        role_df_complete,
        x_col="num_peers_vallen",
        y_col="bytes_sent_percent",
        cat_col="typ",
        filename=f"outputs/{notebook_name}/geobft-spread-over-azs-bytes-sent-role1-{cluster_num_peers}.tex",
        xlabel="(Number of Peers, Vallen)",
        ylabel="Bytes Sent Percent",
        bar=True,
        stack=True,
        symbolic_x=True,
        sort_x_key=lambda x: (int(x.split(",")[0][1:]),int(x.split(",")[1]),str(x.split(",")[2:-1])),
        output_dir=f"outputs/{notebook_name}/"
    )


/mnt/c/Users/Andre/OneDrive/Cloud-Native Byzantine Consensus/code/CloudModuBFT/src/evaluation/notebooks/utils/benchmark.py:372: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



Cluster Num Peers: 3.0
[np.str_('(512,2,False)'), np.str_('(512,2,True)'), np.str_('(512,4,False)'), np.str_('(512,4,True)'), np.str_('(512,6,False)'), np.str_('(512,6,True)'), np.str_('(4096,2,False)'), np.str_('(4096,2,True)'), np.str_('(4096,4,False)'), np.str_('(4096,4,True)'), np.str_('(4096,6,False)'), np.str_('(4096,6,True)')]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode

(./outputs/02-c5.large-geobft-no-cluster-commit/geobft-spread-over-azs-bytes-se
nt-role2-3.0.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/s

/mnt/c/Users/Andre/OneDrive/Cloud-Native Byzantine Consensus/code/CloudModuBFT/src/evaluation/notebooks/utils/benchmark.py:372: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



entering extended mode

(./outputs/02-c5.large-geobft-no-cluster-commit/geobft-spread-over-azs-bytes-se
nt-role1-3.0.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cfg)
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2023/05/17 v1.4n Standard LaTeX document class
(/usr/share/texlive/t

/mnt/c/Users/Andre/OneDrive/Cloud-Native Byzantine Consensus/code/CloudModuBFT/src/evaluation/notebooks/utils/benchmark.py:372: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



entering extended mode

(./outputs/02-c5.large-geobft-no-cluster-commit/geobft-spread-over-azs-bytes-se
nt-role2-6.0.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cfg)
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2023/05/17 v1.4n Standard LaTeX document class
(/usr/share/texlive/t

/mnt/c/Users/Andre/OneDrive/Cloud-Native Byzantine Consensus/code/CloudModuBFT/src/evaluation/notebooks/utils/benchmark.py:372: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode

(./outputs/02-c5.large-geobft-no-cluster-commit/geobft-spread-over-azs-bytes-se
nt-role1-6.0.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cfg)
(/usr/share/texlive/t

In [19]:
res = {}
array_length = 1000
throughputs = process_throughput_benchmarks(benchmarks)
for benchmark in benchmarks: 

    received = np.zeros(array_length)
    response = np.zeros(array_length)
    latency = np.zeros(array_length)
    num_leaders = 0 
    for log in glob(f"{benchmark}/*.log"): 
        
        with open(log, 'r') as f:
            lines = [line for line in f.read().splitlines() if "client request" in line]

        if len(lines) != 2*array_length : 
            continue
        if lines : 
           
            for line in lines : 
                nanoseconds = line.split(" ")[-1]
                if "Received" in line: 
                    rc = int(line.split(" ")[3])
                    # print(line)
                    received[rc] = int(nanoseconds)
                    rc += 1
                elif "Response" in line: 
                    rp = int(line.split(" ")[3])
                    response[rp] = int(nanoseconds)
                    rp += 1
                else: 
                    raise ValueError("Unexpected line")
            
            latency += response - received
            num_leaders+=1
    latency = latency / num_leaders
    latency_ms = latency * 1e-6
    max_latency = np.max(latency_ms)
    avg_latency = np.median(latency_ms)
    min_latency = np.min(latency_ms)
    throughputs.loc[throughputs['benchmark'] == benchmark, 'latency_ms'] = avg_latency
    throughputs.loc[throughputs['benchmark'] == benchmark, 'max_latency_ms'] = max_latency
    throughputs.loc[throughputs['benchmark'] == benchmark, 'min_latency_ms'] = min_latency
    
    if min_latency < 0 : 
        raise ValueError("Negative latency detected")
    print(f"Benchmark: {benchmark}, Max Latency: {max_latency}, Avg Latency: {avg_latency}, Min Latency: {min_latency}")
        

KeyboardInterrupt: 

In [ ]:
throughputs = add_combined_column(throughputs, ['number_of_clusters', 'vallen',"nonoptimistic"], 'num_clusters_vallen')
throughputs["cluster_num_peers"] = throughputs["num_peers"] /  throughputs["number_of_clusters"]
throughputs = throughputs.query("cluster_batch_size < 256")
def generate_latency_plot(throughputs, num_peers,vallen):
    xs = []
    ys = []
    cats = []

    for (group, df) in throughputs.query("cluster_num_peers == @num_peers and vallen == @vallen").groupby("num_clusters_vallen"):
        num_clusters_vallen = group
        sorteddf = df.sort_values(by=["cluster_batch_size", "nonoptimistic", "cluster_num_peers"])
        cats.append(num_clusters_vallen)
        xs.append(sorteddf["cluster_batch_size"].tolist())
        ys.append(sorteddf["latency_ms"].tolist())

    tikz_plot_latency = TikzPlotGenerator(
        xs=xs,
        ys=ys,
        cat=cats,
        xlabel="Cluster Batch Size",
        ylabel="Latency (ms)",
        filename=f"outputs/{notebook_name}/geobft-spread-over-azs-latency-{num_peers}peers-{vallen}bytes.tex",
    )
    tikz_plot_latency.save()
    tikz_plot_latency.compile(output_dir=f"outputs/{notebook_name}/")

    return tikz_plot_latency

generate_latency_plot(throughputs, 6,512)
generate_latency_plot(throughputs, 6,4096)
generate_latency_plot(throughputs, 3,512)
generate_latency_plot(throughputs, 3,4096)

#     throughputs,
#     y_col="latency_ms",
#     x_col="num_peers",
#     cat_col="vallen",
#     filename=f"outputs/{notebook_name}/geobft-spread-over-azs-latency.tex",
#     xlabel="Number of Peers",
#     ylabel="Latency (ms)",
#     output_dir=f"outputs/{notebook_name}/"
# )

[np.int64(1), np.int64(16), np.int64(64), np.int64(128)]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode

(./outputs/02-c5.large-geobft-no-cluster-commit/geobft-spread-over-azs-latency-
6peers-512bytes.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dis

In [ ]:
#rename every file in output_dir=f"outputs/{notebook_name}/"
import os
output_dir=f"outputs/{notebook_name}/"
for filename in os.listdir(output_dir):
    if filename.endswith(".tex") or filename.endswith(".pdf"):
        new_filename = filename.replace("geobft-spread-over-azs","nonoptimistic-geobft-spread-over-azs")
        os.rename(os.path.join(output_dir, filename), os.path.join(output_dir, new_filename))
        
# remove log and aux 
for filename in os.listdir(output_dir):
    if filename.endswith(".log") or filename.endswith(".aux"):
        os.remove(os.path.join(output_dir, filename))

In [ ]:
timestamp=datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

with zipfile.ZipFile(f'outputs/{notebook_name}/notebook_files_{timestamp}.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(f'outputs/{notebook_name}'):
        for file in files:
            if not file.endswith('.zip'):
                zipf.write(os.path.join(root, file), 
                           os.path.relpath(os.path.join(root, file), 
                                           os.path.join(f'outputs/{notebook_name}', '..')))

In [ ]:

s = ""
for filename in os.listdir(output_dir):
    if filename.endswith(".tex"):
        tex_content = "%"+filename+"\n"
        with open(os.path.join(output_dir, filename), 'r') as f:
            for line in f:
                tex_content += line
                
            

        s+= "\n"+tex_content+"\n"

with open(f"outputs/{notebook_name}/figures.tex", 'w') as f:
    f.write(s)